<a href="https://colab.research.google.com/github/GuFerreiraV/sentiment-analysis-research-tgi/blob/main/Notebook_An%C3%A1lise_de_Emo%C3%A7%C3%B5es.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Preparando ambiente
Instalando bibliotecas, modelos e configurando o google drive para persistência de imagens e vídeos e gerando arquivo md com instruções para modelo generativo.

### Instalação de dependências

In [ ]:
!pip install ultralytics
!pip install deepface
!pip install langchain-ollama
!pip install --upgrade ultralytics
!pip install fpdf

In [ ]:
!sudo apt-get install zstd
!sudo apt update
!sudo apt install -y pciutils
!curl -fsSL https://ollama.com/install.sh | sh

### Google Drive

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
caminho_projeto = '/content/drive/MyDrive/notebookcolab/images'

if not os.path.exists(caminho_projeto):
    os.makedirs(caminho_projeto, exist_ok=True)
    print(f"Pasta criada com sucesso em: {caminho_projeto}")
else:
    print(f"A pasta já existe em: {caminho_projeto}")

### Arquivo de instrução

In [ ]:
conteudo_md = """### INSTRUÇÕES

# Persona e Contexto Operacional
Você é um **Módulo Analítico de Fusão Multimodal**. Sua função é receber dados estruturados de Visão Computacional (objetos e expressões faciais) e combiná-los com legendas de texto (quando fornecidas) para gerar um relatório técnico consolidado de análise da cena.

## Diretrizes de Comportamento
1. **Tom de Voz**: Estritamente técnico, objetivo e conciso.
2. **Escopo de Domínio**: Limite-se a analisar sentimentos, interações entre objetos e o contexto da cena.
3. **Restrição de Criatividade**: Proibido inventar elementos não presentes no JSON de entrada. Se a legenda diz "praia" mas o YOLO não detectou areia ou mar, priorize os dados visuais confirmados. Se não houver legenda, baseie a análise puramente nas evidências visuais confirmadas.
4. **Vocabulário Sugerido**: Utilize termos como "correlação", "inferência", "predomínio emocional", "co-ocorrência de objetos".
5. **Expansão Analítica**: Descreva a correlação técnica entre os objetos (YOLO) e a emoção (DeepFace). Se houver dissonância textual, justifique-a com base nas evidências visuais.

## Protocolo de Resolução de Conflitos
Em casos de discrepância entre os Dados Visuais (CV) e a Legenda Original, siga estas regras:
1. **Prioridade de Detecção**: Considere as detecções do YOLO e DeepFace como fatos observados (evidência física).
2. **Contextualização Textual**: Considere a legenda como a intenção ou o sentimento subjetivo do autor.
3. **Relato de Discrepância**: Se houver uma contradição clara (ex: Legenda feliz vs. Rosto triste), o modelo DEVE apontar a "Dissonância Semântica".
4. **Ausência de Legenda**: Caso não seja fornecida uma legenda, avalie a coerência da cena isoladamente (emoção versus ambiente/objetos).

## Estrutura de Saída Obrigatória
Toda resposta deve seguir estritamente o formato abaixo. Não inclua saudações, introduções ou conclusões fora das seções. Use exatamente estes cabeçalhos em colchetes para cada seção:

[1. SUMÁRIO EXECUTIVO]
[Forneça um resumo geral rápido e objetivo da análise da cena]

[2. DETECÇÃO FÍSICA DE AMBIENTE E OBJETOS]
[Descreva os objetos detectados pelo YOLOv8, sua relevância e a inferência de ambiente que eles sugerem]

[3. ANÁLISE DO CLIMA EMOCIONAL]
[Descreva as expressões faciais e humor identificados pelo DeepFace, indicando os percentuais de confiança]

[4. CRUZAMENTO SEMÂNTICO E LEGENDA]
[Realize a correlação entre os dados físicos da imagem e a intenção da legenda fornecida, descrevendo se há concordância ou dissonância semântica]

[5. CONCLUSÃO TÉCNICA]
[Apresente o parecer final técnico unificando todas as evidências analisadas na cena]

## Exemplos de Referência (Few-Shot)

### EXEMPLO 1: CONCORDÂNCIA SEMÂNTICA
- **ENTRADA DE CV**: Objetos: [cachorro, grama, bola]. Emoções: [felicidade].
- **LEGENDA**: "Dia de brincar no parque!"
- **SAÍDA**:
[1. SUMÁRIO EXECUTIVO]
A análise indica um cenário de atividade de lazer ao ar livre em perfeita concordância com a legenda do usuário.

[2. DETECÇÃO FÍSICA DE AMBIENTE E OBJETOS]
Foi registrada a co-ocorrência de um cachorro, grama e uma bola de brinquedo. A presença de vegetação (grama) e o brinquedo indicam com alta probabilidade um ambiente externo recreativo, como um parque ou quintal.

[3. ANÁLISE DO CLIMA EMOCIONAL]
A biometria facial detectou uma expressão facial de felicidade, corroborando o estado de ânimo descontraído.

[4. CRUZAMENTO SEMÂNTICO E LEGENDA]
Concordância semântica total. O ambiente de lazer detectado visualmente reforça o sentido de comemoração e lazer expresso no texto "brincar no parque".

[5. CONCLUSÃO TÉCNICA]
O relatório valida a cena como autêntica e condizente com a legenda informada. Os dados visuais de objetos e biometria dão suporte completo à narrativa textual.
"""

# Define o caminho onde o script orquestrador vai buscar
caminho_instrucoes = '/content/instructions.md'

with open(caminho_instrucoes, 'w', encoding='utf-8') as f:
    f.write(conteudo_md)

print(f"Arquivo {caminho_instrucoes} criado com sucesso!")

In [ ]:
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['TF_USE_LEGACY_KERAS'] = '1'
from google.colab import drive
drive.mount('/content/drive')

### Ollama

In [ ]:
import subprocess
import time

# Tenta encerrar instâncias anteriores para evitar conflitos
!pkill ollama

with open("ollama.log", "w") as log_file:
    process = subprocess.Popen(["ollama", "serve"], stdout=log_file, stderr=log_file)

print("⏳ Aguardando o servidor Ollama iniciar...")
time.sleep(10)

!curl -s http://localhost:11434/api/tags > /dev/null && echo "Servidor Ollama online!" || echo "Falha ao iniciar o servidor."

In [ ]:
!ollama pull gemma2:2b

# Desenvolvimento

1. **Camada de Visão (YOLOv8):** Responsável por identificar objetos físicos (pessoas, carros, acessórios). Se a confiança for baixa, o sistema aplica uma Redução Dinâmica para tentar capturar contexto relevante sem gerar falsos positivos.

2. **Camada Biométrica (DeepFace):** Focada em analisar as faces detectadas utilizando o backend retinaface. O sistema calcula a média emocional do grupo para definir o 'clima' da cena.

3. **Camada de Fusão (LLM):** O script orquestrador une as detecções visuais com a legenda do usuário e submete ao modelo Gemma:2b através do LangChain, buscando identificar concordâncias ou dissonâncias semânticas.

## YOLOv8

In [ ]:
from ultralytics import YOLO

model_yolo_nano = YOLO('yolov8n.pt')

# Usando stream=True para evitar que os resultados se acumulem na RAM e causem crash em vídeos
results = model_yolo_nano.predict(source=caminho_projeto, conf=0.25, stream=True)

# Como results agora é um gerador, precisamos iterar para executar a predição
for r in results:
    pass

print("Varredura inicial concluída com sucesso (Modo Stream).")

### Contagem de objetos

In [ ]:
from collections import Counter

class ObjectDetected:
    def __init__(self, name, confidence):
        self.name = name
        self.confidence = confidence

# Função para extrair os dados brutos do YOLO e converter para sua estrutura
def formatar_objetos_yolo(resultados_yolo):
    # Filtrar por confiança (limite de 50%)
    objetos_filtrados = [
        obj.name for obj in resultados_yolo
        if obj.confidence >= 0.5
    ]

    # Se a lista estiver vazia, aplicar Redução Dinâmica (30%)
    if not objetos_filtrados:
        objetos_filtrados = [
            obj.name for obj in resultados_yolo
            if obj.confidence >= 0.3
        ]
        status = " (Confiança Reduzida)"
    else:
        status = ""

    if not objetos_filtrados:
        return "Nenhum objeto relevante detectado."

    # Contar ocorrências
    contagem = Counter(objetos_filtrados)

    # Montar a string descritiva
    itens = [f"{qtd} {nome}" for nome, qtd in contagem.items()]
    descricao = ", ".join(itens)

    return f"Objetos detectados{status}: {descricao}."

def extrair_dados_yolo(caminho_imagem, modelo):
    # Adicionando parâmetros para evitar NMS Time Limit e gerenciar memória
    # max_det=100 limita o número de caixas para evitar sobrecarga no processamento de sobreposição
    resultados_brutos_gen = modelo.predict(
        source=caminho_imagem,
        conf=0.3,
        verbose=False,
        stream=True,
        max_det=100
    )

    lista_convertida = []
    todos_resultados = []

    for r in resultados_brutos_gen:
        todos_resultados.append(r)
        nomes_classes = r.names

        for box in r.boxes:
            id_classe = int(box.cls[0])
            conf = float(box.conf[0])
            nome = nomes_classes[id_classe]

            lista_convertida.append(ObjectDetected(nome, conf))

    return lista_convertida, todos_resultados

In [ ]:
modelo_nano = YOLO('yolov8n.pt')

# Busca a primeira imagem JPG ou PNG na pasta do projeto para testar com segurança
import glob
imagens_disponiveis = glob.glob(os.path.join(caminho_projeto, '*.jpg')) + glob.glob(os.path.join(caminho_projeto, '*.png'))

if imagens_disponiveis:
  caminho_imagem_teste = imagens_disponiveis[0]
  print(f"Testando inferência YOLOv8 com a imagem: {caminho_imagem_teste}")
  dados_brutos, _ = extrair_dados_yolo(caminho_imagem_teste, modelo_nano)
  string_para_o_prompt = formatar_objetos_yolo(dados_brutos)
  print(string_para_o_prompt)
else:
  print("Nenhuma imagem para teste encontrada na pasta do projeto.")

## DeepFace

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

import tensorflow as tf
# Força o TensorFlow (usado pelo DeepFace) a rodar estritamente na CPU,
tf.config.set_visible_devices([], 'GPU')

from deepface import DeepFace
import cv2

def extracao_de_emocoes(caminho_imagem):
    try:
        # Detecção inicial
        # O DeepFace retorna uma lista de resultados
        resultados = DeepFace.analyze(
            img_path=caminho_imagem,
            actions=['emotion'],
            enforce_detection=False # Evita erro se nenhuma face for encontrada
        )

        if not resultados:
            return "Nenhuma expressão facial detectada."

        face = resultados[0]
        emocao_detectada = face['dominant_emotion']
        confianca = face['emotion'][emocao_detectada] / 100 # O DeepFace fornece a pontuação de todas as emoções; pegamos a dominante

        # Verificação de Limite Fixo (0.5 ou 50%)
        if confianca >= 0.5:
            return f"Emoção: {emocao_detectada} (Alta Confiança: {confianca:.2%})"

        elif confianca >= 0.3:
            return f"Emoção: {emocao_detectada} (Confiança Reduzida: {confianca:.2%} - Tratar como hipótese)"

        else:
            return "Nenhuma expressão facial detectada com clareza suficiente."

    except Exception as e:
        return f"Erro no processamento de imagem: {str(e)}"

### Múltiplas *faces*

In [ ]:
from deepface import DeepFace
from collections import Counter

def analisar_clima_do_grupo(caminho_imagem, detector_backend='retinaface'):
    try:
        resultados = DeepFace.analyze(
            img_path=caminho_imagem,
            actions=['emotion'],
            enforce_detection=False,
            detector_backend=detector_backend
        )

        if not isinstance(resultados, list) or len(resultados) == 0:
            return "Nenhuma expressão facial detectada para análise de grupo.", []

        num_faces = len(resultados)
        soma_emocoes = {'angry': 0, 'disgust': 0, 'fear': 0, 'happy': 0, 'sad': 0, 'surprise': 0, 'neutral': 0}

        for face in resultados:
            if 'emotion' in face:
                for emocao, percentual in face['emotion'].items():
                    soma_emocoes[emocao] += percentual

        # Média das emoções
        media_emocoes = {e: (v / num_faces) for e, v in soma_emocoes.items()}

        top_emocoes = sorted(media_emocoes.items(), key=lambda item: item[1], reverse=True)[:2]

        emocao_1, val_1 = top_emocoes[0]
        emocao_2, val_2 = top_emocoes[1]

        desc = f"Expressões Faciais Predominantes: {emocao_1} ({val_1:.1f}%) e {emocao_2} ({val_2:.1f}%) baseado em {num_faces} faces."
        return desc, resultados

    except Exception as e:
        return f"Erro ao processar clima do grupo: {str(e)}", []

## Montagem de prompt e comunicação com LLM

In [ ]:
import re

def montagem_de_prompt(caminho_instrucoes, info_yolo, info_deepface, legenda_usuario):
    # Sanitização básica para evitar injeção de prompt
    legenda_sanitizada = str(legenda_usuario).replace('"', '\\"').replace('\n', ' ')
    legenda_sanitizada = re.sub(r'[<>{}\[\]]', '', legenda_sanitizada)

    try:
        with open(caminho_instrucoes, 'r') as f:
            instrucoes = f.read()
            # Validação de integridade das tags que definimos
            if "### INSTRUÇÕES" not in instrucoes:
                raise ValueError("Arquivo de instruções inválido: Tag '### INSTRUÇÕES' não encontrada.")
    except FileNotFoundError:
        return "ERRO: O arquivo instructions.md não foi encontrado no caminho especificado."

    super_prompt = f"{instrucoes}\n\n"
    super_prompt += "### DADOS DA IMAGEM\n"
    super_prompt += f"{info_yolo}\n"
    super_prompt += f"{info_deepface}\n\n"
    super_prompt += f"### LEGENDA ORIGINAL\n\"{legenda_sanitizada}\"\n\n"
    super_prompt += "### ANÁLISE TÉCNICA\n"

    return super_prompt

In [ ]:
# Comunicação entre meu prompt com a LLM, usando langchain_chain (minha ponte)
def gerar_analise_tecnica(prompt_completo, langchain_chain):
    resposta = langchain_chain.invoke({"question": prompt_completo})
    return resposta

In [ ]:
from langchain_ollama.llms import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate

# Configuração do Modelo Local (Ollama)
llm_model = OllamaLLM(model="gemma2:2b")

# Definição do Template
template = """Responda à pergunta baseada nas instruções técnicas fornecidas.

Pergunta: {question}

Resposta:"""

prompt_template = ChatPromptTemplate.from_template(template)

# Criação da Chain
chain = prompt_template | llm_model

print("✅ Variável 'chain' inicializada com o modelo gemma:2b!")

## Função de limpeza de gpu

Esta célula irá limpar todo lixo deixado pelo python, além de limpar a sesão do Keras e liberar RAM.

In [ ]:
import torch
import gc
import tensorflow as tf

def limpar_memoria_gpu():
  gc.collect()
  tf.keras.backend.clear_session()
  if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Conclusão

1. **Automação de Relatórios:** A capacidade de transformar dados brutos de visão computacional em documentos PDF técnicos, prontos para análise acadêmica.

2. **Confiabilidade Multimodal:** A integração entre YOLOv8 e DeepFace provou ser eficaz para capturar a complexidade de cenas reais, identificando discrepâncias entre o que é visto e o que é relatado (legenda).

3. **Eficiência Operacional:** Com a implementação de rotinas de limpeza de memória e orquestração via LangChain, o sistema mantém estabilidade mesmo no processamento de lotes de imagens, garantindo a viabilidade do TCC em hardware acessível.

## Configurando relatório

In [ ]:
from fpdf import FPDF
import os
from datetime import datetime

def limpar_analise_llm(analise_texto):
    if not analise_texto:
        return ""
    texto_limpo = analise_texto.strip()
    if texto_limpo.startswith("```"):
        first_line_end = texto_limpo.find("\n")
        if first_line_end != -1:
            texto_limpo = texto_limpo[first_line_end:].strip()
        if texto_limpo.endswith("```"):
            texto_limpo = texto_limpo[:-3].strip()
    try:
        import json as json_lib
        dados = json_lib.loads(texto_limpo)
        if isinstance(dados, list):
            dicionario_unificado = {}
            for item in dados:
                if isinstance(item, dict):
                    dicionario_unificado.update(item)
            dados = dicionario_unificado
        if isinstance(dados, dict):
            linhas_reconstruidas = []
            for chave, valor in dados.items():
                chave_formatada = str(chave).strip()
                chave_formatada = chave_formatada.replace('[', '').replace(']', '')
                linhas_reconstruidas.append(f"[{chave_formatada}]")
                linhas_reconstruidas.append(str(valor).strip())
                linhas_reconstruidas.append("")
            return "\n".join(linhas_reconstruidas)
    except Exception:
        pass
    return analise_texto

def gerar_pdf_consolidado(lista_resultados_processamento, nome_arquivo_final):
    class PDF(FPDF):
        def header(self):
            # Cabeçalho estilizado do Relatório
            self.set_font('Arial', 'B', 12)
            self.set_text_color(30, 60, 120) # Azul escuro acadêmico
            self.cell(0, 10, 'RELATÓRIO TÉCNICO DE ANÁLISE MULTIMODAL', 0, 1, 'L')

            self.set_draw_color(30, 60, 120)
            self.set_line_width(0.5)
            self.line(10, 18, 200, 18)
            self.set_line_width(0.2)
            self.line(10, 19, 200, 19)
            self.ln(6)

        def footer(self):
            self.set_y(-15)
            self.set_font('Arial', 'I', 8)
            self.set_text_color(128, 128, 128)
            data_atual = datetime.now().strftime('%d/%m/%Y %H:%M')
            self.cell(100, 10, f'Módulo Analítico Multimodal TCC | Gerado em {data_atual}', 0, 0, 'L')
            self.cell(0, 10, f'Página {self.page_no()}', 0, 0, 'R')

    pdf = PDF()
    pdf.set_auto_page_break(auto=True, margin=20)

    if not lista_resultados_processamento:
        pdf.add_page()
        pdf.set_font('Arial', '', 12)
        pdf.cell(0, 10, 'Nenhum resultado de imagem para gerar o relatório.', 0, 1, 'C')
    else:
        for resultado in lista_resultados_processamento:
            pdf.add_page()

            # Título da Imagem / Frame
            pdf.set_font('Arial', 'B', 12)
            pdf.set_text_color(50, 50, 50)
            nome_img_display = os.path.basename(resultado['caminho_imagem']) if resultado['caminho_imagem'] else "N/A"
            pdf.cell(0, 8, f'Mídia Analisada: {nome_img_display}', 0, 1, 'L')
            pdf.ln(2)

            # Inserir Imagem Anotada
            img_path = resultado['caminho_anotado']
            if img_path and os.path.exists(img_path):
                try:
                    # Centralizar imagem A4
                    pdf.image(img_path, x=45, y=pdf.get_y(), w=120)
                    pdf.ln(95)
                except Exception as e:
                    pdf.set_font('Arial', 'I', 10)
                    pdf.set_text_color(200, 50, 50)
                    pdf.cell(0, 10, f'[Erro ao carregar referência visual: {e}]', 0, 1, 'C')
                    pdf.ln(5)
            else:
                # Caso não haja imagem anotada
                pdf.set_fill_color(250, 240, 240)
                pdf.set_draw_color(220, 180, 180)
                current_y = pdf.get_y()
                pdf.rect(10, current_y, 190, 12, 'DF')
                pdf.set_xy(12, current_y + 3)
                pdf.set_font('Arial', 'I', 10)
                pdf.set_text_color(150, 50, 50)
                pdf.cell(0, 5, 'Referência visual anotada não disponível para esta mídia.', 0, 1)
                pdf.ln(8)

            # Processar e estilizar o texto da LLM
            analise_texto = resultado['analise_llm']
            analise_texto = limpar_analise_llm(analise_texto)

            # Limpar formatações de Markdown comuns
            analise_texto = analise_texto.replace('**', '')
            analise_texto = analise_texto.replace('```json', '')
            analise_texto = analise_texto.replace('```', '')

            # Tratar caracteres para Latin-1
            try:
                analise_texto = analise_texto.encode('latin-1', 'replace').decode('latin-1')
            except Exception:
                analise_texto = analise_texto.encode('ascii', 'ignore').decode('ascii')

            linhas = analise_texto.split('\n')

            for linha in linhas:
                linha = linha.strip()
                if not linha:
                    pdf.ln(2)
                    continue

                # Identificar delimitador de seção
                if linha.startswith('[') and ']' in linha:
                    titulo_secao = linha.replace('[', '').replace(']', '')
                    pdf.ln(4)
                    pdf.set_font('Arial', 'B', 10)
                    pdf.set_text_color(30, 60, 120)
                    pdf.cell(0, 5, titulo_secao, 0, 1, 'L')
                    pdf.ln(1)
                else:
                    # Linha de conteúdo normal
                    pdf.set_font('Arial', '', 9.5)
                    pdf.set_text_color(40, 40, 40)
                    pdf.multi_cell(0, 4.5, linha)
                    pdf.ln(1)

    caminho_pdf = os.path.join(caminho_projeto, f'{nome_arquivo_final}.pdf')
    pdf.output(caminho_pdf)
    print(f'✅ PDF Relatório Consolidado gerado em: {caminho_pdf}')
    return caminho_pdf

## Pipeline CI/CD

In [ ]:
import os
import glob
import cv2
from tqdm import tqdm

def salvar_visualizacao_deteccao(caminho_img, resultados_yolo, resultados_deepface, pasta_saida='/content/outputs'):
    img = cv2.imread(caminho_img)
    if img is None:
        print(f"  -> Erro ao carregar imagem para anotação: {caminho_img}")
        return None

    os.makedirs(pasta_saida, exist_ok=True)

    # Bounding Boxes
    for r in resultados_yolo:
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            conf = float(box.conf[0])
            cls_id = int(box.cls[0])
            cls_name = r.names[cls_id]
            if conf >= 0.3:
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                label = f"YOLO: {cls_name} {conf:.2f}"
                cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 1)

    # DeepFace Bounding Boxes
    if isinstance(resultados_deepface, list):
        for face in resultados_deepface:
            if 'region' in face:
                region = face['region']
                x, y, w, h = region['x'], region['y'], region['w'], region['h']
                cv2.rectangle(img, (x, y), (x+w, y+h), (0, 0, 255), 2)
                if 'dominant_emotion' in face:
                    emocao = face['dominant_emotion']
                    conf_emocao = face['emotion'][emocao]
                    label_emocao = f"Face: {emocao} ({conf_emocao:.1f}%)"
                    cv2.putText(img, label_emocao, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    nome_saida = os.path.basename(caminho_img)
    caminho_saida = os.path.join(pasta_saida, f"debug_{nome_saida}")
    cv2.imwrite(caminho_saida, img)
    return caminho_saida

def executar_pipeline(caminho_img, modelo_yolo, modelo_llm, caminho_regras, legenda=None, detector_backend='retinaface'):
    print(f"[{os.path.basename(caminho_img)}] Iniciando processamento...")

    texto_legenda = legenda if legenda and str(legenda).strip() != "" else "Nenhuma legenda fornecida para esta imagem."

    # Visão Computacional
    dados_yolo, resultados_yolo = extrair_dados_yolo(caminho_img, modelo_yolo)
    info_yolo = formatar_objetos_yolo(dados_yolo)
    info_deepface, resultados_deepface = analisar_clima_do_grupo(caminho_img, detector_backend=detector_backend)

    # Salvar imagem anotada com Bounding Boxes
    caminho_anotado = None
    try:
        caminho_anotado = salvar_visualizacao_deteccao(caminho_img, resultados_yolo, resultados_deepface, '/content/outputs')
        if caminho_anotado:
            print(f"  -> Imagem com Bounding Boxes salva em: {caminho_anotado}")
    except Exception as e:
        print(f"  -> Erro ao desenhar Bounding Boxes: {e}")

    # Orquestração de Prompt
    prompt = montagem_de_prompt(caminho_regras, info_yolo, info_deepface, texto_legenda)

    # Inferência LLM
    print("  -> Analisando contexto multimodal com LLM...")
    analise = gerar_analise_tecnica(prompt, modelo_llm)

    # Limpeza de memória
    limpar_memoria_gpu()

    print("  -> Finalizado e Memória Limpa!")
    return {
        'caminho_imagem': caminho_img,
        'caminho_anotado': caminho_anotado,
        'info_yolo': info_yolo,
        'info_deepface': info_deepface,
        'legenda': texto_legenda,
        'analise_llm': analise
    }

def processar_lote_imagens(pasta_imagens, modelo_yolo, modelo_llm, caminho_regras, nome_pdf_final="Relatorio_Consolidado_Multimodal", detector_backend='retinaface'):
    extensoes = ('*.jpg', '*.jpeg', '*.png', '*.webp')
    lista_imagens = []

    for ext in extensoes:
        lista_imagens.extend(glob.glob(os.path.join(pasta_imagens, ext)))

    if not lista_imagens:
        print(f"ATENÇÃO: Nenhuma imagem encontrada na pasta: {pasta_imagens}")
        return

    print(f"Encontradas {len(lista_imagens)} imagens para processar.")

    resultados_para_pdf = []
    for caminho_img in tqdm(lista_imagens, desc="Processando Imagens"):
        nome_arquivo = os.path.basename(caminho_img)
        try:
            resultado_imagem = executar_pipeline(
                caminho_img=caminho_img,
                legenda=None,
                modelo_yolo=modelo_yolo,
                modelo_llm=modelo_llm,
                caminho_regras=caminho_regras,
                detector_backend=detector_backend
            )
            resultados_para_pdf.append(resultado_imagem)
        except Exception as e:
            print(f"  -> Erro crítico ao processar o arquivo '{nome_arquivo}': {e}")

    print("\nProcessamento de imagens concluído. Gerando relatório consolidado...")
    gerar_pdf_consolidado(resultados_para_pdf, nome_pdf_final)
    print("Todos os lotes foram processados com sucesso!")

print("INICIANDO VARREDURA DA PASTA...")
processar_lote_imagens(
    pasta_imagens=caminho_projeto,
    modelo_yolo=model_yolo_nano,
    modelo_llm=chain,
    caminho_regras=caminho_instrucoes
)

### Análise de Vídeo
Esta seção permite processar arquivos de vídeo (.mp4, .avi, etc), extraindo frames periodicamente para análise multimodal.

In [ ]:
import cv2
import os

def processar_video(caminho_video, modelo_yolo, modelo_llm, caminho_regras, intervalo_segundos=2, detector_backend='opencv'):
    """
    Extrai frames de um vídeo e processa cada um através do pipeline com limpeza agressiva de memória.
    """
    if not os.path.exists(caminho_video):
        print(f"Erro: Vídeo não encontrado em {caminho_video}")
        return

    video = cv2.VideoCapture(caminho_video)
    fps = video.get(cv2.CAP_PROP_FPS)
    intervalo_frames = int(fps * intervalo_segundos)

    resultados_video = []
    frame_count = 0
    sucesso = True

    nome_video = os.path.splitext(os.path.basename(caminho_video))[0]
    print(f"Iniciando análise otimizada do vídeo: {nome_video}")

    while sucesso:
        sucesso, frame = video.read()

        if sucesso and frame_count % intervalo_frames == 0:
            caminho_temp_frame = f"/content/temp_frame_{frame_count}.jpg"
            cv2.imwrite(caminho_temp_frame, frame)

            segundo = int(frame_count/fps)
            print(f"\n--- Analisando Segundo {segundo} ---")

            try:
                resultado = executar_pipeline(
                    caminho_img=caminho_temp_frame,
                    modelo_yolo=modelo_yolo,
                    modelo_llm=modelo_llm,
                    caminho_regras=caminho_regras,
                    detector_backend=detector_backend
                )
                resultado['caminho_imagem'] = f"{nome_video} (Segundo {segundo})"
                resultados_video.append(resultado)
            except Exception as e:
                print(f"Erro no frame {frame_count}: {e}")

            # Limpeza imediata pós-frame
            if os.path.exists(caminho_temp_frame):
                os.remove(caminho_temp_frame)

            # Força a limpeza de cache e lixo de memória para evitar o crash
            limpar_memoria_gpu()

        frame_count += 1

    video.release()

    if resultados_video:
        print(f"\nAnálise concluída. Gerando relatório para {len(resultados_video)} frames...")
        gerar_pdf_consolidado(resultados_video, f"Relatorio_Video_{nome_video}")
        # Limpeza final
        limpar_memoria_gpu()
    else:
        print("Nenhum frame foi processado com sucesso.")

### Executar Análise de Vídeo
Use a célula abaixo para processar vídeos encontrados na pasta do projeto.

In [ ]:
import glob

# Busca arquivos de vídeo comuns na pasta do projeto
extensoes_video = ('*.mp4', '*.avi', '*.mov', '*.mkv')
lista_videos = []

for ext in extensoes_video:
    lista_videos.extend(glob.glob(os.path.join(caminho_projeto, ext)))

if not lista_videos:
    print(f"Nenhum vídeo encontrado em: {caminho_projeto}")
else:
    print(f"Encontrado(s) {len(lista_videos)} vídeo(s). Iniciar processamento?")
    for video_path in lista_videos:
        # Processa o vídeo (ajuste o intervalo_segundos se necessário)
        processar_video(
            caminho_video=video_path,
            modelo_yolo=model_yolo_nano,
            modelo_llm=chain,
            caminho_regras=caminho_instrucoes,
            intervalo_segundos=5 # Analisa 1 frame a cada 5 segundos para maior rapidez
        )